# 04 — Factorized Neural-Circuit Model

This notebook demonstrates the Phase 2 upgrade: a **4-factor generative model** where each hidden state factor maps to a distinct neural circuit.

| Factor | States | Neural proxy | Role |
|--------|--------|-------------|------|
| S_identity | {anonymous, partial, full} | **TPJ** (mentalizing) | Victim individuation |
| S_affect | {low, medium, high} | **Insula** (affective salience) | Empathic arousal |
| S_distance | {proximal, distal, abstract} | **mPFC** (self-other distance) | Psychological/institutional distance |
| S_outcome | {not_saved_nocost, not_saved_cost, saved_cost} | **Striatum** (valuation) | Action outcomes |

**IVE mechanism**: Identity modulates the precision of affect observations (A_affect matrix). When a victim is fully identified, the affect signal is sharper, producing stronger empathic responses that bias toward helping.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

from ive.networks import (
    build_network_agent, choose_network_action,
    get_network_help_probability, context_to_network_states,
    apply_aggregation, CASE_PRESETS, NETWORK_DEFAULTS,
    IDENTITY_LABELS, AFFECT_LABELS, DISTANCE_LABELS, OUTCOME_LABELS,
    OBS_IDENTITY, OBS_AFFECT, OBS_DISTANCE, OBS_OUTCOME, OBS_COST,
)

## 1. Model structure inspection

Build agents for statistical and identified contexts and inspect the expected free energy.

In [ ]:
for ctx in ['stat', 'id', 'high_id']:
    states = context_to_network_states(ctx)
    agent = build_network_agent(**states)
    agent.reset()
    obs = [states['identity_state'], states['affect_state'],
           states['distance_state'], 0, 0]
    agent.infer_states(obs)
    agent.infer_policies()
    
    print(f'Context: {ctx}')
    print(f'  States: identity={IDENTITY_LABELS[states["identity_state"]]}, '
          f'affect={AFFECT_LABELS[states["affect_state"]]}, '
          f'distance={DISTANCE_LABELS[states["distance_state"]]}')
    print(f'  G (EFE): NoHelp={agent.G[0]:.3f}, Help={agent.G[1]:.3f}')
    print(f'  q_pi:    P(NoHelp)={agent.q_pi[0]:.3f}, P(Help)={agent.q_pi[1]:.3f}')
    print()

## 2. Monte Carlo: IVE in the factorized model

In [ ]:
np.random.seed(42)
n = 800

help_rates = {}
for ctx in ['stat', 'id', 'high_id']:
    states = context_to_network_states(ctx)
    p = get_network_help_probability(n_samples=n, **states)
    help_rates[ctx] = p
    print(f'P(Help|{ctx:7s}) = {p:.3f}')

print(f'\nIVE (id - stat):      {help_rates["id"] - help_rates["stat"]:+.3f}')
print(f'IVE (high_id - stat): {help_rates["high_id"] - help_rates["stat"]:+.3f}')

fig, ax = plt.subplots(figsize=(5, 4))
contexts = ['Statistical', 'Identified', 'Highly Id.']
vals = [help_rates['stat'], help_rates['id'], help_rates['high_id']]
colors = ['C0', 'C1', 'C2']
ax.bar(range(3), vals, color=colors, alpha=0.8)
ax.set_xticks(range(3))
ax.set_xticklabels(contexts)
ax.set_ylim(0, 1)
ax.set_ylabel('P(Help)')
ax.set_title('Factorized model: graded IVE')
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Precision modulation: the core IVE mechanism

The key computational mechanism is that **identity modulates the precision of affect representations**. Let's visualise how the A_affect matrix changes across identity and distance states.

In [ ]:
# Build an agent and extract A_affect
agent = build_network_agent(identity_state=0, affect_state=1, distance_state=0)
A_affect = agent.A[OBS_AFFECT]  # shape: (3, 3, 3, 3, 3)

# Extract effective precision for affect_state=1 (medium)
# across identity x distance conditions
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

for s_id in range(3):
    for s_dist in range(3):
        ax = axes[s_id, s_dist]
        # P(obs_affect | s_affect) for this identity x distance
        # A_affect shape: (obs_affect, s_identity, s_affect, s_distance, s_outcome)
        obs_dist = A_affect[:, s_id, :, s_dist, 0]  # (3 obs, 3 affect states)
        
        im = ax.imshow(obs_dist, cmap='Blues', vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(3))
        ax.set_xticklabels(AFFECT_LABELS, fontsize=8)
        ax.set_yticks(range(3))
        ax.set_yticklabels(AFFECT_LABELS, fontsize=8)
        
        if s_dist == 0:
            ax.set_ylabel(f'Id: {IDENTITY_LABELS[s_id]}', fontsize=9)
        if s_id == 0:
            ax.set_title(f'Dist: {DISTANCE_LABELS[s_dist]}', fontsize=9)
        
        # Show diagonal precision
        diag_prec = np.mean([obs_dist[i, i] for i in range(3)])
        ax.text(1, -0.6, f'prec={diag_prec:.2f}', ha='center', fontsize=8,
                transform=ax.transData)

fig.suptitle('A_affect: P(obs_affect | s_affect) across identity x distance\n'
             '(rows: identity level, cols: distance level)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Summary: effective affect precision as a function of identity and distance
precisions = np.zeros((3, 3))
for s_id in range(3):
    for s_dist in range(3):
        obs_dist = A_affect[:, s_id, :, s_dist, 0]
        precisions[s_id, s_dist] = np.mean([obs_dist[i, i] for i in range(3)])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(precisions, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='auto')
ax.set_xticks(range(3))
ax.set_xticklabels(DISTANCE_LABELS)
ax.set_yticks(range(3))
ax.set_yticklabels(IDENTITY_LABELS)
ax.set_xlabel('Distance (mPFC)')
ax.set_ylabel('Identity (TPJ)')
ax.set_title('Effective affect precision (Insula)\n'
             'Identity x Distance interaction')
plt.colorbar(im, ax=ax, label='Affect observation precision')

for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{precisions[i, j]:.2f}', ha='center', va='center',
                fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Parameter sweeps: each neural circuit's contribution

In [ ]:
np.random.seed(42)
n_mc = 400

# Sweep identity_affect_coupling
coupling_vals = np.linspace(0.0, 1.5, 10)
h_stat_c, h_id_c = [], []
for c in coupling_vals:
    ps = get_network_help_probability(
        n_samples=n_mc, identity_affect_coupling=c,
        **context_to_network_states('stat'))
    pi = get_network_help_probability(
        n_samples=n_mc, identity_affect_coupling=c,
        **context_to_network_states('high_id'))
    h_stat_c.append(ps)
    h_id_c.append(pi)

# Sweep distance_affect_attenuation
atten_vals = np.linspace(0.0, 1.0, 10)
h_stat_d, h_id_d = [], []
for a in atten_vals:
    ps = get_network_help_probability(
        n_samples=n_mc, distance_affect_attenuation=a,
        **context_to_network_states('stat'))
    pi = get_network_help_probability(
        n_samples=n_mc, distance_affect_attenuation=a,
        **context_to_network_states('high_id'))
    h_stat_d.append(ps)
    h_id_d.append(pi)

# Sweep identity_precision
id_prec_vals = np.linspace(0.1, 0.99, 10)
h_stat_p, h_id_p = [], []
for p in id_prec_vals:
    ps = get_network_help_probability(
        n_samples=n_mc, identity_precision=p,
        **context_to_network_states('stat'))
    pi = get_network_help_probability(
        n_samples=n_mc, identity_precision=p,
        **context_to_network_states('high_id'))
    h_stat_p.append(ps)
    h_id_p.append(pi)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, xvals, stat, iden, xlabel, title in [
    (axes[0], coupling_vals, h_stat_c, h_id_c,
     'identity_affect_coupling', 'TPJ-Insula coupling'),
    (axes[1], atten_vals, h_stat_d, h_id_d,
     'distance_affect_attenuation', 'mPFC-Insula attenuation'),
    (axes[2], id_prec_vals, h_stat_p, h_id_p,
     'identity_precision', 'TPJ precision'),
]:
    ax.plot(xvals, stat, 'o-', label='Statistical', color='C0')
    ax.plot(xvals, iden, 's-', label='Highly identified', color='C2')
    ax.fill_between(xvals, stat, iden, alpha=0.15, color='C1')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('P(Help)')
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Parameter sweeps: neural circuit contributions to the IVE', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Comparison with Phase 1 single-factor model

In [ ]:
from ive.agent import get_help_probability

np.random.seed(42)
n_cmp = 1000

# Phase 1 model (best-fit params from notebook 02)
p1_stat = get_help_probability(
    delta_C=0.9, delta_p=0.1, cost_penalty=1.9,
    context='stat', n_samples=n_cmp)
p1_id = get_help_probability(
    delta_C=0.9, delta_p=0.1, cost_penalty=1.9,
    context='id', n_samples=n_cmp)

# Phase 2 factorized model
p2_stat = get_network_help_probability(
    n_samples=n_cmp, **context_to_network_states('stat'))
p2_id = get_network_help_probability(
    n_samples=n_cmp, **context_to_network_states('high_id'))

print(f'{"Model":25s}  P(stat)  P(id)   IVE')
print('-' * 55)
print(f'{"Phase 1 (single factor)":25s}  {p1_stat:.3f}    {p1_id:.3f}   {p1_id - p1_stat:+.3f}')
print(f'{"Phase 2 (4 factors)":25s}  {p2_stat:.3f}    {p2_id:.3f}   {p2_id - p2_stat:+.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
width = 0.2
ax.bar(x - width*1.5, [p1_stat, p2_stat], width, label='Statistical', color='C0', alpha=0.8)
ax.bar(x - width*0.5, [p1_id, p2_id], width, label='Identified', color='C2', alpha=0.8)
ive1 = p1_id - p1_stat
ive2 = p2_id - p2_stat
ax.bar(x + width*0.5, [ive1, ive2], width, label='IVE magnitude', color='C3', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(['Phase 1\n(single factor)', 'Phase 2\n(4 factors)'])
ax.set_ylim(0, 0.8)
ax.set_ylabel('P(Help) / IVE')
ax.set_title('Model comparison: Phase 1 vs Phase 2')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Neural circuit schematic

The factorized model implements the following information flow:

```
S_identity (TPJ)  ----precision modulation----> A_affect (Insula)
                                                    |
S_distance (mPFC) ----attenuation--------------> A_affect
                                                    |
                                                    v
                                            Affect preference
                                                    |
                                                    v
S_outcome (Striatum) <--- policy selection <--- Expected Free Energy
                                                    ^
                                                    |
                                            C_outcome (util_saved)
                                            C_cost   (-cost_penalty)
```

**Key predictions for neuroimaging (Phase 3):**

| Prediction | Neural signature |
|------------|------------------|
| Identified > Statistical | TPJ activation (identity encoding) |
| Identified -> stronger affect | Insula activation (affect precision) |
| Aggregation -> reduced help | Reduced TPJ-Insula functional connectivity |
| Distance -> reduced help | mPFC activation (abstract processing) |
| Policy precision | dACC / Striatal activation (decision urgency) |